In [ ]:
# Jupyter Notebook
print("OK")
# Importando as bibliotecas necessárias
import os
import json
import subprocess
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Configurações iniciais
sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None  # Para evitar warnings de cópias de DataFrame

# Configurando os diretórios
input_dir = Path("runs")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)


## Execução da Simulação

In [ ]:
def main():
    # Activate virtual environment (if needed)
    venv_path = os.path.join(os.getcwd(), "venv")
    if os.path.exists(venv_path):
        activate_script = os.path.join(venv_path, "bin", "activate_this.py")
        with open(activate_script) as f:
            exec(f.read(), {"__file__": activate_script})

    # Definir app_dir FIXO (correto)
    app_dir = "/home/nathan/home/UFCG/gEco/Projetos/IPD/ipd/ipd"
    config_json_file = os.path.join(app_dir, "config.json")

    # Ler config
    with open(config_json_file) as config_file:
        config = json.load(config_file)
        kernel_dir = config["paths"]["kernel"]
        runs_dir = config["paths"]["runs"]
        model_file = config["files"]["model"]
        scenarios_file = config["files"]["scenarios"]

    # Limpar CSVs antigos
    if os.path.exists(runs_dir):
        for file in os.listdir(runs_dir):
            if file.endswith(".csv"):
                os.remove(os.path.join(runs_dir, file))

    # Guardar diretório atual (Jupyter)
    original_dir = os.getcwd()

    # Rodar simulação
    os.chdir(kernel_dir)
    subprocess.run(
        ["python3", "ecosimp.py", app_dir, model_file, scenarios_file],
        check=True
    )

    # Voltar para o diretório original
    os.chdir(original_dir)

if __name__ == "__main__":
    main()

## Análise do Resultado

In [ ]:
# Configurações iniciais
sns.set(style="whitegrid")
pd.options.mode.chained_assignment = None  # Evita warnings desnecessários

# Config the paths
input_dir = Path("/home/nathan/home/UFCG/gEco/Projetos/IPD/ipd/ipd/runs")

output_dir = Path("/home/nathan/home/UFCG/gEco/Projetos/IPD/ipd/ipd/results")
output_dir.mkdir(exist_ok=True)

# CSV
csv_files = list(input_dir.glob("*.csv"))

# Verificando se existem arquivos CSV
if not csv_files:
    raise ValueError(f"Nenhum arquivo CSV foi encontrado em: {input_dir.resolve()}")

# Gerando o DataFrame consolidado
runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
runs_df_or = runs_df.copy()
runs_df = runs_df.dropna()

# Calculando a diferença de payoffs
runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# Agrupando os dados para calcular médias e somas de payoffs
sum_runs = (
    runs_df.groupby(["scenario", "step", "strategy_name"])
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_sum=("my_payoff", "sum"),
    )
    .reset_index()
)

# Calculando cooperação e defecção
sum_cooperation = (
    runs_df.groupby(["scenario", "step"])
    .agg(
        my_play_C=("my_play", lambda x: (x == "C").sum()),
        my_play_D=("my_play", lambda x: (x == "D").sum()),
        ot_play_C=("other_play", lambda x: (x == "C").sum()),
        ot_play_D=("other_play", lambda x: (x == "D").sum()),
        players=("my_play", "count"),
    )
    .reset_index()
)

sum_cooperation["cooperation"] = sum_cooperation["my_play_C"] / sum_cooperation["players"]
sum_cooperation["defection"] = sum_cooperation["my_play_D"] / sum_cooperation["players"]

# Tabela com média e mediana por agente
agent_summary = (
    runs_df.groupby("strategy_name")
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_median=("my_payoff", "median"),
        payoff_sum=("my_payoff", "sum"),
        observations=("my_payoff", "count"),
    )
    .reset_index()
    .sort_values(by="payoff_mean", ascending=False)
)

# Exibindo a tabela
print("Tabela de média e mediana por agente:")
display(agent_summary)

# Salvando a tabela
agent_summary.to_csv(output_dir / "agent_summary.csv", index=False)

# Gera uma paleta mais saturada
unique_strategies = sum_runs["strategy_name"].unique()
palette = sns.color_palette("bright", n_colors=len(unique_strategies))

# Cria um mapeamento de cores
color_dict = dict(zip(unique_strategies, palette))

# Gráfico 1: Payoff médio por estratégia ao longo do tempo
g = sns.lmplot(
    data=sum_runs, 
    x="step", 
    y="payoff_mean",
    hue="strategy_name",
    palette=color_dict,
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.7},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

# Títulos e eixos
g.set_axis_labels("Passo", "Payoff Médio")
g.fig.suptitle("Payoff Médio por Estratégia ao Longo do Tempo", fontsize=14, y=1.03)

# Remove a legenda original
if g._legend is not None:
    g._legend.remove()

# Cria legenda personalizada com patches coloridos
handles = [Patch(color=color_dict[strategy], label=strategy) for strategy in unique_strategies]
g.ax.legend(handles=handles, title="Estratégia", fontsize=10, title_fontsize=11)

# Salvar e mostrar
plt.savefig(output_dir / "payoff_mean_by_strategy.png", bbox_inches="tight")
plt.show()

# Gráfico 2: Distribuição de payoff médio por estratégia
plt.figure(figsize=(10, 6))
sns.violinplot(data=sum_runs, x="strategy_name", y="payoff_mean", inner=None)
sns.boxplot(data=sum_runs, x="strategy_name", y="payoff_mean", width=0.2, color="white")
sns.pointplot(
    data=sum_runs,
    x="strategy_name",
    y="payoff_mean",
    color="red",
    join=False,
    markers="o",
    scale=1.5,
)
plt.title("Distribuição de Payoff Médio por Estratégia")
plt.xlabel("Estratégia")
plt.ylabel("Payoff Médio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / "payoff_distribution_by_strategy.png")
plt.show()

# Gráfico 3A: Cooperação ao longo do tempo (dispersão)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    alpha=0.5
)
plt.title("Cooperação ao Longo do Tempo")
plt.xlabel("Passo")
plt.ylabel("Cooperação")
plt.savefig(output_dir / "cooperation_scatter.png")
plt.show()

# Gráfico 3B: Cooperação ao longo do tempo com suavização
g = sns.lmplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.5},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

g.set_axis_labels("Passo", "Cooperação")
g.fig.suptitle("Cooperação ao Longo do Tempo", fontsize=14, y=1.03)

plt.savefig(output_dir / "cooperation_over_time.png", bbox_inches="tight")
plt.show()

In [ ]:

# # Obtendo os arquivos CSV
# csv_files = list(input_dir.glob("*.csv"))

# # Gerando o DataFrame consolidado
# runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
# runs_df_or = runs_df.copy()
# runs_df = runs_df.dropna()

# # Calculando a diferença de payoffs
# runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# # Agrupando os dados para calcular médias e somas de payoffs
# sum_runs = (
#     runs_df.groupby(["scenario", "step", "strategy_name"])
#     .agg(
#         payoff_mean=("my_payoff", "mean"),
#         payoff_sum=("my_payoff", "sum"),
#     )
#     .reset_index()
# )

# # Calculando cooperação e defecção
# sum_cooperation = (
#     runs_df.groupby(["scenario", "step"])
#     .agg(
#         my_play_C=("my_play", lambda x: (x == "C").sum()),
#         my_play_D=("my_play", lambda x: (x == "D").sum()),
#         ot_play_C=("other_play", lambda x: (x == "C").sum()),
#         ot_play_D=("other_play", lambda x: (x == "D").sum()),
#         players=("my_play", "count"),
#     )
#     .reset_index()
# )

# sum_cooperation["cooperation"] = sum_cooperation["my_play_C"] / sum_cooperation["players"]
# sum_cooperation["defection"] = sum_cooperation["my_play_D"] / sum_cooperation["players"]


# from matplotlib.patches import Patch

# # Gera uma paleta mais saturada
# unique_strategies = sum_runs["strategy_name"].unique()
# palette = sns.color_palette("bright", n_colors=len(unique_strategies))

# # Cria um mapeamento de cores
# color_dict = dict(zip(unique_strategies, palette))

# # Recria o lmplot com a paleta
# g = sns.lmplot(
#     data=sum_runs, 
#     x="step", 
#     y="payoff_mean",
#     hue="strategy_name",
#     palette=color_dict,
#     scatter=True, 
#     lowess=True, 
#     line_kws={"alpha": 0.7},
#     scatter_kws={"alpha": 0.1, "s": 10},
#     height=6, aspect=1.5
# )

# # Títulos e eixos
# g.set_axis_labels("Passo", "Payoff Médio")
# g.fig.suptitle("Payoff Médio por Estratégia ao Longo do Tempo", fontsize=14, y=1.03)

# # Remove a legenda original
# g._legend.remove()

# # Cria legenda personalizada com patches coloridos e mais saturados
# handles = [Patch(color=color_dict[strategy], label=strategy) for strategy in unique_strategies]
# g.ax.legend(handles=handles, title="Estratégia", fontsize=10, title_fontsize=11)

# # Salvar e mostrar
# plt.savefig(output_dir / "payoff_mean_by_strategy.png", bbox_inches="tight")
# plt.show()

# # Gráfico 2: Distribuição de payoff médio por estratégia
# plt.figure(figsize=(10, 6))
# sns.violinplot(data=sum_runs, x="strategy_name", y="payoff_mean", inner=None)
# sns.boxplot(data=sum_runs, x="strategy_name", y="payoff_mean", width=0.2, color="white")
# sns.pointplot(
#     data=sum_runs,
#     x="strategy_name",
#     y="payoff_mean",
#     color="red",
#     join=False,
#     markers="o",
#     scale=1.5,
# )
# plt.title("Distribuição de Payoff Médio por Estratégia")
# plt.xlabel("Estratégia")
# plt.ylabel("Payoff Médio")
# plt.savefig(output_dir / "payoff_distribution_by_strategy.png")
# plt.show()

# # Gráfico 3: Cooperação ao longo do tempo
# plt.figure(figsize=(10, 6))
# sns.scatterplot(data=sum_cooperation, 
#                 x="step", 
#                 y="cooperation", 
#                 alpha=0.5
#                 )

# plt.figure(figsize=(10, 6))
# sns.lmplot(data=sum_cooperation, 
#            x="step", 
#            y="cooperation", 
#            scatter=True, 
#             lowess=True, 
#             line_kws={"alpha": 0.5},
#             scatter_kws={"alpha": 0.1, "s": 10}
#             )

# plt.title("Cooperação ao Longo do Tempo")
# plt.xlabel("Passo")
# plt.ylabel("Cooperação")
# plt.savefig(output_dir / "cooperation_over_time.png")
# plt.show()

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from pathlib import Path

# # =========================
# # CONFIGURAÇÕES VISUAIS
# # =========================
# sns.set_theme(style="whitegrid", context="talk")
# plt.rcParams["figure.dpi"] = 120
# plt.rcParams["savefig.dpi"] = 300

# # =========================
# # LEITURA DOS DADOS
# # =========================
# csv_files = list(input_dir.glob("*.csv"))

# if not csv_files:
#     raise ValueError("Nenhum arquivo CSV foi encontrado em input_dir.")

# runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
# runs_df_or = runs_df.copy()

# # Remova NA apenas nas colunas essenciais da análise
# cols_essenciais = [
#     "scenario", "step", "strategy_name",
#     "my_payoff", "other_payoff",
#     "my_play", "other_play"
# ]
# runs_df = runs_df.dropna(subset=cols_essenciais).copy()

# # Garantir tipos corretos
# runs_df["step"] = pd.to_numeric(runs_df["step"], errors="coerce")
# runs_df["my_payoff"] = pd.to_numeric(runs_df["my_payoff"], errors="coerce")
# runs_df["other_payoff"] = pd.to_numeric(runs_df["other_payoff"], errors="coerce")
# runs_df = runs_df.dropna(subset=["step", "my_payoff", "other_payoff"])

# # Diferença de payoff
# runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# # =========================
# # AGREGAÇÕES
# # =========================
# sum_runs = (
#     runs_df.groupby(["scenario", "step", "strategy_name"], as_index=False)
#     .agg(
#         payoff_mean=("my_payoff", "mean"),
#         payoff_median=("my_payoff", "median"),
#         payoff_sum=("my_payoff", "sum"),
#         payoff_std=("my_payoff", "std"),
#         diff_mean=("diff_payoffs", "mean"),
#         n_obs=("my_payoff", "size")
#     )
# )

# sum_cooperation = (
#     runs_df.groupby(["scenario", "step"], as_index=False)
#     .agg(
#         my_play_C=("my_play", lambda x: (x == "C").sum()),
#         my_play_D=("my_play", lambda x: (x == "D").sum()),
#         ot_play_C=("other_play", lambda x: (x == "C").sum()),
#         ot_play_D=("other_play", lambda x: (x == "D").sum()),
#         players=("my_play", "count"),
#     )
# )

# sum_cooperation["cooperation"] = sum_cooperation["my_play_C"] / sum_cooperation["players"]
# sum_cooperation["defection"] = sum_cooperation["my_play_D"] / sum_cooperation["players"]

# # =========================
# # TABELA-RESUMO POR ESTRATÉGIA
# # =========================
# strategy_summary = (
#     runs_df.groupby("strategy_name", as_index=False)
#     .agg(
#         media_payoff=("my_payoff", "mean"),
#         mediana_payoff=("my_payoff", "median"),
#         desvio_padrao=("my_payoff", "std"),
#         media_diff=("diff_payoffs", "mean"),
#         mediana_diff=("diff_payoffs", "median"),
#         observacoes=("my_payoff", "size")
#     )
#     .sort_values("media_payoff", ascending=False)
# )

# # Arredondar para visualização
# strategy_summary_round = strategy_summary.copy()
# for col in ["media_payoff", "mediana_payoff", "desvio_padrao", "media_diff", "mediana_diff"]:
#     strategy_summary_round[col] = strategy_summary_round[col].round(3)

# print("\nTabela resumo por estratégia:")
# print(strategy_summary_round)

# # Salvar tabela
# strategy_summary_round.to_csv(output_dir / "tabela_resumo_estrategias.csv", index=False)

# # Ordem das estratégias pela mediana
# strategy_order = (
#     strategy_summary.sort_values("mediana_payoff", ascending=False)["strategy_name"]
#     .tolist()
# )

# # =========================
# # GRÁFICO 1: EVOLUÇÃO DO PAYOFF MÉDIO
# # =========================
# plt.figure(figsize=(14, 8))

# sns.lineplot(
#     data=sum_runs,
#     x="step",
#     y="payoff_mean",
#     hue="strategy_name",
#     estimator=None,
#     alpha=0.30,
#     linewidth=1,
#     legend=False
# )

# # suavização por estratégia
# for strategy in strategy_order:
#     df_strat = (
#         sum_runs[sum_runs["strategy_name"] == strategy]
#         .sort_values("step")
#         .copy()
#     )

#     # média móvel para suavizar
#     window = max(5, int(len(df_strat["step"].unique()) * 0.03))
#     df_strat["smooth"] = df_strat["payoff_mean"].rolling(window=window, min_periods=1).mean()

#     plt.plot(
#         df_strat["step"],
#         df_strat["smooth"],
#         linewidth=2.5,
#         label=strategy
#     )

# plt.title("Evolução do Payoff Médio por Estratégia")
# plt.xlabel("Passo")
# plt.ylabel("Payoff médio")
# plt.legend(title="Estratégia", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.savefig(output_dir / "01_payoff_medio_evolucao.png", bbox_inches="tight")
# plt.show()

# # =========================
# # GRÁFICO 2: DISTRIBUIÇÃO DOS PAYOFFS
# # =========================
# plt.figure(figsize=(14, 8))

# sns.violinplot(
#     data=runs_df,
#     x="strategy_name",
#     y="my_payoff",
#     order=strategy_order,
#     inner=None,
#     cut=0
# )

# sns.boxplot(
#     data=runs_df,
#     x="strategy_name",
#     y="my_payoff",
#     order=strategy_order,
#     width=0.20,
#     showcaps=True,
#     boxprops={"facecolor": "white", "zorder": 3},
#     whiskerprops={"linewidth": 1.2},
#     medianprops={"color": "black", "linewidth": 2}
# )

# plt.title("Distribuição dos Payoffs por Estratégia")
# plt.xlabel("Estratégia")
# plt.ylabel("Payoff")
# plt.xticks(rotation=45, ha="right")
# plt.tight_layout()
# plt.savefig(output_dir / "02_distribuicao_payoffs.png", bbox_inches="tight")
# plt.show()

# # =========================
# # GRÁFICO 3: MÉDIA E MEDIANA POR ESTRATÉGIA
# # =========================
# summary_melt = strategy_summary_round.melt(
#     id_vars="strategy_name",
#     value_vars=["media_payoff", "mediana_payoff"],
#     var_name="medida",
#     value_name="valor"
# )

# plt.figure(figsize=(12, 7))

# sns.barplot(
#     data=summary_melt,
#     x="strategy_name",
#     y="valor",
#     hue="medida",
#     order=strategy_order
# )

# plt.title("Média e Mediana do Payoff por Estratégia")
# plt.xlabel("Estratégia")
# plt.ylabel("Valor")
# plt.xticks(rotation=45, ha="right")
# plt.legend(title="")
# plt.tight_layout()
# plt.savefig(output_dir / "03_media_mediana_por_estrategia.png", bbox_inches="tight")
# plt.show()

# # =========================
# # GRÁFICO 4: COOPERAÇÃO AO LONGO DO TEMPO
# # =========================
# coop_time = (
#     sum_cooperation.groupby("step", as_index=False)
#     .agg(
#         cooperation=("cooperation", "mean"),
#         defection=("defection", "mean")
#     )
#     .sort_values("step")
# )

# window_coop = max(5, int(len(coop_time) * 0.03))
# coop_time["cooperation_smooth"] = coop_time["cooperation"].rolling(window=window_coop, min_periods=1).mean()
# coop_time["defection_smooth"] = coop_time["defection"].rolling(window=window_coop, min_periods=1).mean()

# plt.figure(figsize=(14, 8))

# plt.plot(coop_time["step"], coop_time["cooperation"], alpha=0.25, linewidth=1, label="Cooperação (bruto)")
# plt.plot(coop_time["step"], coop_time["cooperation_smooth"], linewidth=3, label="Cooperação (suavizada)")

# plt.plot(coop_time["step"], coop_time["defection"], alpha=0.20, linewidth=1, linestyle="--", label="Defecção (bruto)")
# plt.plot(coop_time["step"], coop_time["defection_smooth"], linewidth=3, linestyle="--", label="Defecção (suavizada)")

# plt.title("Cooperação e Defecção ao Longo do Tempo")
# plt.xlabel("Passo")
# plt.ylabel("Proporção")
# plt.ylim(0, 1)
# plt.legend()
# plt.tight_layout()
# plt.savefig(output_dir / "04_cooperacao_defeccao_tempo.png", bbox_inches="tight")
# plt.show()

# # =========================
# # GRÁFICO 5: MEDIANA DO PAYOFF POR ESTRATÉGIA
# # =========================
# plt.figure(figsize=(12, 7))

# sns.barplot(
#     data=strategy_summary_round,
#     x="strategy_name",
#     y="mediana_payoff",
#     order=strategy_order
# )

# plt.title("Mediana do Payoff por Estratégia")
# plt.xlabel("Estratégia")
# plt.ylabel("Mediana do payoff")
# plt.xticks(rotation=45, ha="right")
# plt.tight_layout()
# plt.savefig(output_dir / "05_mediana_payoff_estrategia.png", bbox_inches="tight")
# plt.show()